예제 1.

In [2]:
class MyStack:
  def pop(self):
    return "Removed"

def remove_last(obj):
  return obj.pop()

print(remove_last([1, 2, 3]))
print(remove_last(MyStack()))

3
Removed


예제 2.

In [3]:
class MySequence:
  def __init__(self, data):
    self.data = data

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    return self.data[index]

seq = MySequence([10, 20, 30])

print(len(seq))
print(seq[1])

for item in seq:
  print(item)

3
20
10
20
30


예제 3.

In [4]:
class Foo:
  def __getitem__(self, pos):
    return range(0, 30, 10)[pos]

In [7]:
f = Foo()
f[1]

10

In [8]:
for i in f: print(i)

0
10
20


In [9]:
20 in f

True

In [10]:
15 in f

False

예제 4.

In [11]:
PyObject *
PyObject_GetIter(PyObject *o)
{
   PyTypeObject *t = Py_TYPE(o);
   getiterfunc f;

   f = t->tp_iter;
   if (f == NULL) {
       if (PySequence_Check(o))
           return PySeqIter_New(o);
       return type_error("'%.200s' object is not iterable", o);
   }
   else {
       PyObject *res = (*f)(o);
       if (res != NULL && !PyIter_Check(res)) {
                   PyErr_Format(PyExc_TypeError,
                        "%T.__iter__() must return an iterator, not %T",
                        o, res);
           Py_SETREF(res, NULL);
       }
       return res;
    }
}

SyntaxError: invalid syntax (4249924944.py, line 1)

예제 5.

In [23]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
  ranks =  [str(n) for n in range(2, 11)] + list('JQKA')
  suits = 'spades diamonds clubs hearts'.split()

  def __init__(self):
    self._cards = [Card(rank, suit) for suit in self.suits
                                    for rank in self.ranks]

  def __len__(self):
    return len(self._cards)

  def __getitem__(self, position):
    return self._cards[position]

예제 6.

In [24]:
from random import shuffle
deck = FrenchDeck()
shuffle(deck)

TypeError: 'FrenchDeck' object does not support item assignment

예제 7.

In [26]:
def set_card(deck, position, card):
  deck._cards[position] = card

FrenchDeck.__setitem__ = set_card
shuffle(deck)
deck[:5]

[Card(rank='2', suit='spades'),
 Card(rank='8', suit='spades'),
 Card(rank='A', suit='spades'),
 Card(rank='Q', suit='spades'),
 Card(rank='7', suit='clubs')]

예제 8.

In [30]:
from collections.abc import MutableSequence
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck2(MutableSequence):
  ranks =  [str(n) for n in range(2, 11)] + list('JQKA')
  suits = 'spades diamonds clubs hearts'.split()

  def __init__(self):
    self._cards = [Card(rank, suit) for suit in self.suits
                                    for rank in self.ranks]

  def __len__(self):
    return len(self._cards)

  def __getitem__(self, position):
    return self._cards[position]

  def __setitem__(self, position, value):
    self._cards[position] = value

  def __delitem__(self, position):
    del self._cards[position]

  def insert(self, position, value):
    self._cards.insert(position, value)

예제 9.

In [31]:
from abc import ABC, abstractmethod

class Drawable(ABC):
  @abstractmethod
  def draw(self):
    pass

class ExternalPainter:
  def draw(self):
    print("Drawing")

Drawable.register(ExternalPainter)

obj = ExternalPainter()

print(isinstance(obj, Drawable))

True


예제 10.

In [32]:
from collections.abc import Sized

class MyCollection:
  def __len__(self):
    return 10

Sized.register(MyCollection)

__main__.MyCollection

예제 11.

In [35]:
import abc

class Tombola(abc.ABC):

  @abc.abstractmethod
  def load(self, iteralbe):
    """Add items from an iterable."""

  @abc.abstractmethod
  def pick(self):
    """Remove item at random, returning it.
    This method should raise 'LookupError' when the instance is empty.
    """

  def loaded(self):
    """Return 'True' if ther's at least 1 item, 'False' oterwise."""
    return bool(self.inspect())

  def inspect(self):
    """Return a sorted tuple with the items currently inside."""
    items = []
    while True:
      try:
        items.append(self.pick())
      except LookupError:
        break
    self.load(items)
    return tuple(sorted(items))

예제 12.

In [38]:
import random

#from tombola import Tombola

class BingoCage(Tombola):
  def __init__(self, items):
    self._randomizer = random.SystemRandom()
    self._items = []
    self.load(items)

  def load(self, items):
    self._items.extend(items)
    self._randomizer.shuffle(self._items)

  def pick(self):
    try:
      return self._items.pop()
    except IndexError:
      raise LookupError('pick from empty BingoCage')

  def __call__(self):
    self.pick()

예제 13.

In [40]:
import random

#from tombola import Tombola

class LotteryBlower(Tombola):

  def __init__(self, iterable):
    self._balls = list(iterable)

  def load(self, iterable):
    self._balls.extend(iterable)

  def pick(self):
    try:
      position = random.randrange(len(self._balls))
    except ValueError:
      raise LookupError('pick from empty BingoCage')
    return self._balls.pop(position)

  def loaded(self):
    return bool(self._balls)

  def inspect(self):
    return tuple(sorted(self._balls))

예제 14.

In [42]:
from random import randrange

#from tombola import Tombola

@Tombola.register
class TomboList(list):

  def pick(self):
    if self:
      position = randrange(len(self))
      return self.pop(position)
    else:
      raise LookupError('pop from empty TomboList')

  load = list.extend

  def loaded(self):
    return bool(self)

  def inspect(self):
    return tuple(sorted(self))

예제 15.

In [45]:
import doctest

#from tombola import Tombola

#import bingo, lotto, tombolist, drum

TEST_FILE = 'tombola_tests.rst'
TEST_MSG = '{0:16} {1.attempted:2} tests, {1.failed:2} failed - {2}'

def main(argv):
  verbose = '-v' in argv
  real_subclasses = Tombola.__subclasses__()
  virtual_subclasses = list(Tombola._abc_registry)

  for cls in real_subclasses + virtual_subclasses:
    test(cls, verbose)

  def test(cls, verbose=False):

    res = doctest.testfile(
            TEST_FILE,
            globs={'ConcreteTombola': cls},
            verbose=verbose,
            optionflags=doctest.REPORT_ONLY_FIRST_FAILURE)
    tag = 'FAIL' if res.failed else 'OK'
    print(TEST_MSG.format(cls.__name__, res, tag))

    if __name__ == '__main__':
      import sys
      main(sys.argv)